In [1]:
import os
import pandas as pd
import re

In [2]:
import os
path = os.path.dirname(os.path.dirname(os.getcwd()))
print(path)

c:\Universidad\1\Universidad\TFG\TFG


In [3]:
metadata = pd.read_excel(path+'/transcripts/Discourse-UWO/metadata.xlsx',index_col=0)

In [4]:
idx_psico= metadata.index[metadata['PatientCat']==2].tolist()

In [5]:
print(f"Transcriptions not processed: {idx_psico[:5]}")

Transcriptions not processed: [7, 12, 13, 15, 16]


In [6]:
def extract_interviews(folder_path,followup=False):
    todas_interviews = {}
    pat = {}
    for filename in os.listdir(folder_path):
        if int(filename.split('.')[0]) not in idx_psico[5:]:
            continue
        if filename.endswith('.cha') and filename not in ["007.cha"]:
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                if followup:
                    filename = filename + "_followup"
                groups = []
                group = []
                content = file.readlines()#[10:]
                content_no_mod = content.copy()
                line_index = [i for i, line in enumerate(content) if line.startswith('@G:	Free')]
                if line_index is None or len(line_index)==0:
                    continue
                else:
                    line_index = line_index[0]
                content = content[line_index+1:]
                for row in content:
                    if row.startswith('@'):
                        if len(group)==0:
                            pass
                        else:
                            group = [re.sub(r'\t',' ',line.strip()) for line in group if line.strip() and not line.startswith('@')]
                            group = [re.sub(r'.*$', '', line) for line in group]
                            group = [line[1:].strip() for line in group if line.startswith('*INV:') or line.startswith('*PAR:')]
                            groups.append(group)
                            group = []
                            if pat.get(filename) is None:
                                pat[filename] = 1
                            else:
                                pat[filename] +=1
                    elif row.startswith('%'):
                        pass
                    else:
                        if pat.get(filename) is None:
                            pat[filename] = 1
                        else:
                            pat[filename] +=1
                        group.append(row.strip())         
                todas_interviews[filename] = groups
    todas_interviews = {k:v for k,v in todas_interviews.items() if len(v)==7}
    return todas_interviews,pat


In [7]:
folder_path = path +'/transcripts/Discourse-UWO/Baseline'
todas_interviews, pat_baseline = extract_interviews(folder_path)
folder_path_followup = path +'/transcripts/Discourse-UWO/FollowUp'
todas_interviews_followup, pat_followup = extract_interviews(folder_path_followup, followup=True)

In [8]:
it_base = list(todas_interviews.items())
it_followup = list(todas_interviews_followup.items())
it_base.extend(it_followup)
todas_interviews = dict(it_base)


In [9]:
todas_processed = {}
#Lo parseamos en (speaker, utterance)
for file in todas_interviews:
    todas_processed[file] = []
    for i,part in enumerate(todas_interviews[file]):
        aux = [(line[:3],line[4:].strip()) for i, line in enumerate(part)]
        todas_processed[file].append({"part":i, "utterances":aux})
print(f"Ejemplo: {todas_processed[list(todas_processed.keys())[0]][:5]}")

Ejemplo: [{'part': 0, 'utterances': [('INV', 'so &-um I would like you to talk about a few things related to your daily life .'), ('INV', "you don't have to use anyone's names when answering or describing this event ."), ('INV', 'okay ?'), ('INV', 'can you tell me a bit about yourself ?'), ('PAR', 'a bit about myself ?'), ('INV', 'yeah .'), ('PAR', "well I'm [/] I'm [//] I was born in a island called Trinidad And Tobago ."), ('INV', 'mhm .'), ('PAR', 'and I grew up there .'), ('PAR', '&-uh I played &-uh and &-um .'), ('PAR', 'I had a happy childhood .'), ('PAR', 'but &-um <my my> [/] my dad and stuff .'), ('PAR', '&-uh he suffered from alcoholism and stuff .'), ('PAR', 'so anyhow &-um my parents and stuff .'), ('PAR', 'they [/] they got a divorce .'), ('PAR', "and then I that's when I moved to Canada with my mom and stuff ."), ('PAR', 'and &-um we lived in Toronto for a while and it was pretty rough you know like .'), ('PAR', 'in a new country getting situated you know adapting to the 

In [10]:
prep_df = []
for file, group in todas_processed.items():
    for part in group:
        # print(part,file,intervenciones)
        prep_df.extend([(file,part["part"],i[0],i[1]) for i in part["utterances"]])
df = pd.DataFrame(prep_df,columns=['filename','part','role','text'])
df.head(5)

,filename,part,role,text
0,017.cha,0,INV,so &-um I would like you to talk about a few t...
1,017.cha,0,INV,you don't have to use anyone's names when answ...
2,017.cha,0,INV,okay ?
3,017.cha,0,INV,can you tell me a bit about yourself ?
4,017.cha,0,PAR,a bit about myself ?



### Agrupando todas las intervenciones del mismo hablante

In [11]:
todas_agrupadas = {}
for file, partes in todas_processed.items():
    todas_agrupadas[file] = []
    for parte in partes:
        num_parte,texto = parte["part"], parte["utterances"]
        agrupadas = []
        if not texto:
            print(file)
            todas_agrupadas[file] = agrupadas
            continue
        num =0
        actual_speaker, actual_text = texto[0]
        for i, (speaker, text) in enumerate(texto[1:]):
            if speaker == actual_speaker:
                actual_text += "\n " + text
            else:
                agrupadas.append({"position": num, "speaker": actual_speaker, "text": actual_text})
                actual_speaker, actual_text = speaker, text
                num += 1
        agrupadas.append({"position": num, "speaker": actual_speaker, "text": actual_text})  # Añade la última intervención
        todas_agrupadas[file].append({"part": num_parte, "utterances": agrupadas})

In [12]:
prep_df = []
for file, intervenciones in todas_agrupadas.items():
    for parte in intervenciones:
        # print(parte,file,intervenciones)
        prep_df.extend([(file,parte["part"],i["position"],i["speaker"],i["text"]) for i in parte["utterances"]])
df = pd.DataFrame(prep_df,columns=['filename','part','position','role','answer'])
df

,filename,part,position,role,answer
0,017.cha,0,0,INV,so &-um I would like you to talk about a few t...
1,017.cha,0,1,PAR,a bit about myself ?
2,017.cha,0,2,INV,yeah .
3,017.cha,0,3,PAR,well I'm [/] I'm [//] I was born in a island c...
4,017.cha,0,4,INV,mhm .
...,...,...,...,...,...
18022,148.cha_followup,6,2,INV,read [/] read the story out loud from beginnin...
18023,148.cha_followup,6,3,PAR,it was a hot day .\n a thirsty bird was lookin...
18024,148.cha_followup,6,4,INV,perfect .\n now can you tell me that story aga...
18025,148.cha_followup,6,5,PAR,yeah .\n so basically there was like some bird...


In [13]:
df_par = df[df['role'] == 'PAR'].copy()
df_par['question'] = df_par.apply(
    lambda row: df.loc[(df['filename'] == row['filename']) & (df['role'] == 'INV') & (df.index < row.name), 'answer'].iloc[-1]
    if not df.loc[(df['filename'] == row['filename']) & (df['role'] == 'INV') & (df.index < row.name), 'answer'].empty else None,
    axis=1
)
df_par.reset_index(drop=True, inplace=True)


In [ ]:
df_par['patient'] = df_par['filename'].apply(lambda x: int(x.split('.')[0]))
df_par['schizophrenia'] = 1
df_par.sample(5)

,filename,part,position,role,answer,question,patient,schizophrenia
7402,071.cha_followup,0,5,PAR,what I do do ?,awesome .\n so you say you don't &-uh go out a...,71,1
3877,111.cha,0,15,PAR,so it's very stress free compared to other pro...,yeah .,111,1
7944,105.cha_followup,1,7,PAR,&-uh pretty boring .\n I've had lots of appoin...,I'm sorry to hear that &*PAR:mhm .\n &-um anyt...,105,1
134,020.cha,0,3,PAR,I'm xxxX I born nineteen ninety two twenty fiv...,can you tell me a bit about yourself ?,20,1
2487,089.cha,1,13,PAR,I think it is problems by myself you have to d...,yeah .\n it is always sad .\n when people leav...,89,1


In [16]:
import json
data = df_par.to_dict(orient='records')
with open(path +'/data/ordered_patient_discourse_qa.json','w',encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
print(f"Wrote {path}/data/ordered_patient_discourse_qa.json with {len(data)} records")

Wrote c:\Universidad\1\Universidad\TFG\TFG/data/ordered_patient_discourse_qa.json with 8606 records


In [ ]:
from datasets import load_dataset, Dataset
import numpy as np

print("Loading SFT dataset...")

dataset = load_dataset('json', data_files=path+'/data/ordered_patient_discourse_qa.json')

# Añadir columna de paciente para agrupar
df_temp = df_par.copy()

# Obtener lista única de pacientes y hacer shuffle
unique_patients = df_temp['patient'].unique()
np.random.seed(42)
np.random.shuffle(unique_patients)

# Dividir pacientes en train (80%), test (10%), eval (10%)
n_patients = len(unique_patients)
n_train = int(n_patients * 0.8)
n_test = int(n_patients * 0.1)

train_patients = unique_patients[:n_train]
test_patients = unique_patients[n_train:n_train+n_test]
eval_patients = unique_patients[n_train+n_test:]

# Crear datasets basados en pacientes
train_df = df_temp[df_temp['patient'].isin(train_patients)].to_dict('list')
test_df = df_temp[df_temp['patient'].isin(test_patients)].to_dict('list')
eval_df = df_temp[df_temp['patient'].isin(eval_patients)].to_dict('list')

train_dataset_sft = Dataset.from_dict(train_df)
test_dataset_sft = Dataset.from_dict(test_df)
eval_dataset_sft = Dataset.from_dict(eval_df)

train_dataset_sft.to_json(path+'/data/ordered_PT_train_dataset.json')
test_dataset_sft.to_json(path+'/data/ordered_PT_test_dataset.json')
eval_dataset_sft.to_json(path+'/data/ordered_PT_eval_dataset.json')

print("SFT Dataset loaded (grouped by patient):")
print(f" Train samples: {len(train_dataset_sft)} ({len(train_patients)} patients)")
print(f" Test samples: {len(test_dataset_sft)} ({len(test_patients)} patients)")
print(f" Eval samples: {len(eval_dataset_sft)} ({len(eval_patients)} patients)")
print(f"\n Single Sample: {train_dataset_sft[0]}")

c:\Universidad\1\Universidad\TFG\TFG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading SFT dataset...


Generating train split: 8606 examples [00:00, 58001.30 examples/s]
Creating json from Arrow format: 100%|██████████| 1/1 [00:00<?, ?ba/s]

SFT Dataset loaded (grouped by patient):
 Train samples: 7078 (71 patients)
 Test samples: 778 (8 patients)
 Eval samples: 750 (10 patients)

 Single Sample: {'filename': '017.cha', 'part': 0, 'position': 1, 'role': 'PAR', 'answer': 'a bit about myself ?', 'question': "so &-um I would like you to talk about a few things related to your daily life .\n you don't have to use anyone's names when answering or describing this event .\n okay ?\n can you tell me a bit about yourself ?", 'patient': 17, 'schizophrenia': 1}
